# Parameter Optimization

PyBroker v2 supports parameterized strategies. This allows backtesting strategies that use different combinations of parameters and then automatically selecting the best performing parameters. This process is known as **parameter optimization**, and is handled in PyBroker using the [Optuna framework](https://optuna.org/). 

These strategy parameters are created as hyperparameters, as shown in the next section.

## Declaring Hyperparameters

A hyperparameter is a named, tunable value created with `pybroker.hyperparam`. Each one has a `default` that regular backtests use, and a search range given by `low`, `high`, and `step`. The candidate values start at `low` (inclusive), and then increase by `step` until `high` (inclusive).

A hyperparameter can be used to parameterize:

- **Indicators**: pass it as a keyword argument to `pybroker.indicator`.
- **Executions**: attach it with `hyperparams=` on `add_execution` and read it with `ctx.hyperparam` in the execution function.

To demonstrate, we will build a moving average crossover strategy with two hyperparameters: the moving average's `period` and a `stop_pct` stop loss. 

In [1]:
import numpy as np
import pybroker
from pybroker import Strategy, YFinance

pybroker.enable_data_source_cache("parameter_optimization")

period = pybroker.hyperparam("period", default=30, low=10, high=50, step=10)
stop_pct = pybroker.hyperparam(
    "stop_pct", default=6.0, low=2.0, high=10.0, step=2.0
)


def sma(bar_data, period):
    close = bar_data.close
    values = np.full(len(close), np.nan)
    for i in range(period - 1, len(close)):
        values[i] = close[i - period + 1 : i + 1].mean()
    return values


# The hyperparam is passed in place of a concrete period.
sma_ind = pybroker.indicator("sma", sma, period=period)


def sma_cross_stop(ctx):
    sma = ctx.indicator("sma")
    if np.isnan(sma[-1]):
        return
    pos = ctx.long_pos()
    if not pos and ctx.close[-1] > sma[-1]:
        ctx.buy_shares = 100
        ctx.stop_loss_pct = ctx.hyperparam("stop_pct")
    elif pos and ctx.close[-1] < sma[-1]:
        ctx.sell_shares = pos.shares


strategy = Strategy(YFinance(), start_date="1/1/2025", end_date="8/1/2026")
strategy.add_execution(
    sma_cross_stop,
    ["MRK", "TGT", "ORCL"],
    indicators=sma_ind,
    hyperparams=[stop_pct],
)

result = strategy.backtest()
print(f"Total return: {result.metrics.total_return_pct:.2f}%")

Backtesting: 2025-01-01 00:00:00 to 2026-08-01 00:00:00

Loaded cached bar data.

Computing indicators...


100% (3 of 3) |##########################| Elapsed Time: 0:00:00 Time:  0:00:00



Test split: 2025-01-02 00:00:00 to 2026-07-31 00:00:00


100% (395 of 395) |######################| Elapsed Time: 0:00:00 Time:  0:00:00



Finished backtest: 0:00:00
Total return: 1.14%


## Optimizing with Grid Search

The `optimize` method splits the data into train and test windows as specified by `train_size` (`0.5` for 50/50 by default). As Optuna searches the parameter space, each selected combination  is backested on the train window and scored with `score_fn`. By default, this score is maximized (pass `direction="minimize"` to minimize instead). To guard against overfitting, the backtests are run exclusively on the training window. Once finished, the best performing parameters are evaluated on the out-of-sample test window. 

In Optuna, a sampler is the algorithm that decides which parameter combinations to test during the optimization process. The default sampler is "grid", which will evaluate every possible parameter combination. For our example, this results in 5 × 5 = 25 total trials. These trials are evaluated in parallel on the workers configured using [set_parallel](https://www.pybroker.com/en/latest/notebooks/11.%20Configuring%20Parallelization.html):

In [2]:
def score_fn(result):
    return result.metrics.total_return_pct


opt_result = strategy.optimize(score_fn)
print("Best params:", opt_result.best_params)
print("Best train score:", opt_result.best_score)

Loaded cached bar data.

Optimizing: 25 trials (grid)
Computing indicators...


100% (3 of 3) |##########################| Elapsed Time: 0:00:00 Time:  0:00:00



Test split: 2025-10-17 00:00:00 to 2026-07-31 00:00:00


100% (197 of 197) |######################| Elapsed Time: 0:00:00 Time:  0:00:00



Best params: {'period': 20, 'stop_pct': 8.0}
Best train score: 11.105560000000004


`best_params` holds the winning values, and `best_score` is the score they earned on the train window. The `result` attribute contains the `TestResult` of replaying the winning values on the test window:

In [3]:
opt_result.result.metrics_df.head()

,name,value
0,trade_count,34
1,initial_market_value,100000.0
2,end_market_value,98995.24
3,total_pnl,-1782.76
4,unrealized_pnl,778.0


## Optimizing with Tree-structured Parzen Estimator (TPE)

Grid search grows multiplicatively with each hyperparameter added. An alternative approach is using `sampler="tpe"`. Optuna's `TPESampler` (Tree-structured Parzen Estimator) uses [Bayesian Optimization](https://en.wikipedia.org/wiki/Bayesian_optimization), fitting a probability model to completed trials in order to suggest the most promising values for the next run. Note that `n_trials` is required for every sampler except `"grid"`, and providing a seed makes the parameter search reproducible. 

Because TPE adapts based on earlier results, its trials always run sequentially:

In [4]:
opt_result = strategy.optimize(score_fn, sampler="tpe", n_trials=15, seed=2)
print("Best params:", opt_result.best_params)
print("Best train score:", opt_result.best_score)

Loaded cached bar data.

Optimizing: 15 trials (tpe)
Loaded cached indicator data.

Test split: 2025-10-17 00:00:00 to 2026-07-31 00:00:00


100% (197 of 197) |######################| Elapsed Time: 0:00:00 Time:  0:00:00



Best params: {'period': 20, 'stop_pct': 8.0}
Best train score: 11.105560000000004


TPE recovered the same best values as the exhaustive grid search while evaluating only 15 of the 25 combinations.

## Other Samplers

`sampler="random"` chooses combinations uniformly at random with Optuna's [RandomSampler](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.RandomSampler.html). Like grid, random trials can be evaluated in parallel:

In [5]:
opt_result = strategy.optimize(score_fn, sampler="random", n_trials=10, seed=1)
print("Best params:", opt_result.best_params)

Loaded cached bar data.

Optimizing: 10 trials (random)
Computing indicators...


100% (3 of 3) |##########################| Elapsed Time: 0:00:00 Time:  0:00:00



Test split: 2025-10-17 00:00:00 to 2026-07-31 00:00:00


100% (197 of 197) |######################| Elapsed Time: 0:00:00 Time:  0:00:00



Best params: {'period': 20, 'stop_pct': 6.0}


Any [optuna.samplers.BaseSampler](https://optuna.readthedocs.io/en/stable/reference/samplers/index.html) instance can also be passed directly to customize the parameter search:

In [6]:
from optuna.samplers import TPESampler

# Use fewer random startup trials before TPE's model takes over.
opt_result = strategy.optimize(
    score_fn, sampler=TPESampler(n_startup_trials=5), n_trials=15, seed=2
)
print("Best params:", opt_result.best_params)

Loaded cached bar data.

Optimizing: 15 trials (tpe)
Loaded cached indicator data.

Test split: 2025-10-17 00:00:00 to 2026-07-31 00:00:00


100% (197 of 197) |######################| Elapsed Time: 0:00:00 Time:  0:00:00



Best params: {'period': 20, 'stop_pct': 8.0}


Every optimization also returns the underlying [optuna.Study](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html) for inspecting the trials:

In [7]:
opt_result.study.trials_dataframe().head()

,number,value,datetime_start,datetime_complete,duration,params_period,params_stop_pct,state
0,0,1.77596,2026-08-01 14:21:15.684890,2026-08-01 14:21:15.709436,0 days 00:00:00.024546,20,4.0,COMPLETE
1,1,-7.06564,2026-08-01 14:21:15.709468,2026-08-01 14:21:15.730596,0 days 00:00:00.021128,50,2.0,COMPLETE
2,2,-7.06564,2026-08-01 14:21:15.730630,2026-08-01 14:21:15.821854,0 days 00:00:00.091224,50,2.0,COMPLETE
3,3,0.30808,2026-08-01 14:21:15.821887,2026-08-01 14:21:15.842486,0 days 00:00:00.020599,40,6.0,COMPLETE
4,4,-0.38130,2026-08-01 14:21:15.842520,2026-08-01 14:21:15.859477,0 days 00:00:00.016957,50,10.0,COMPLETE


## Walkforward Optimization

Finally, optimize supports walkforward optimization. If you pass `> 1`to the `windows` parameter, PyBroker will independently tune the hyperparameters for each window split and then combine the out-of-sample results:

In [8]:
opt_result = strategy.optimize(score_fn, windows=3)

for i, window in enumerate(opt_result.windows):
    print(
        f"Window {i + 1} train: {window.train_start_date:%Y-%m-%d} to "
        f"{window.train_end_date:%Y-%m-%d}, "
        f"test: {window.test_start_date:%Y-%m-%d} to "
        f"{window.test_end_date:%Y-%m-%d}"
    )
    print(f"Window {i + 1} best params:", window.params)

Loaded cached bar data.

Optimizing: 3 windows, 25 trials per window (75 total, grid)
Computing indicators...


100% (3 of 3) |##########################| Elapsed Time: 0:00:00 Time:  0:00:00



Test split: 2025-05-30 00:00:00 to 2025-10-17 00:00:00


100% (98 of 98) |########################| Elapsed Time: 0:00:00 Time:  0:00:00



Computing indicators...


...


Test split: 2025-10-20 00:00:00 to 2026-03-11 00:00:00


100% (98 of 98) |########################| Elapsed Time: 0:00:00 Time:  0:00:00



Loaded cached indicator data.

Test split: 2026-03-12 00:00:00 to 2026-07-31 00:00:00


100% (98 of 98) |########################| Elapsed Time: 0:00:00 Time:  0:00:00



Window 1 train: 2025-01-07 to 2025-05-29, test: 2025-05-30 to 2025-10-17
Window 1 best params: {'period': 10, 'stop_pct': 2.0}
Window 2 train: 2025-05-30 to 2025-10-17, test: 2025-10-20 to 2026-03-11
Window 2 best params: {'period': 20, 'stop_pct': 8.0}
Window 3 train: 2025-10-20 to 2026-03-11, test: 2026-03-12 to 2026-07-31
Window 3 best params: {'period': 20, 'stop_pct': 6.0}


Each window is tuned separately, so the best parameters can differ between windows. The combined result contains the `best_params` of the **last** (most recent) window:

In [9]:
print("Best params:", opt_result.best_params)

Best params: {'period': 20, 'stop_pct': 6.0}
